**Java 24+:** Spark/Hadoop need Java 17 (older JDKs work too). Run the cell below first — it sets `JAVA_HOME` and `YELP_JAVA17_HOME` from `mise` if installed — then **restart the kernel** and run all cells from the top.

In [1]:
# Run this first if you use Java 25+. Then restart kernel and run all.
import os, subprocess
path = ""
try:
    r = subprocess.run(["mise", "where", "java@17"], capture_output=True, text=True, timeout=5)
    path = (r.stdout or "").strip()
except Exception:
    pass
if path and os.path.exists(os.path.join(path, "bin", "java")):
    os.environ["YELP_JAVA17_HOME"] = path
    os.environ["JAVA_HOME"] = path
    print("Set JAVA_HOME and YELP_JAVA17_HOME to", path, "-- restart the kernel and run all cells.")
else:
    print("Install Java 17 first: run in terminal:  mise install java@17")

Set JAVA_HOME and YELP_JAVA17_HOME to /home/dash/.local/share/mise/installs/java/17.0.2 -- restart the kernel and run all cells.


In [2]:
import importlib

from IPython.display import display

from src.constants import PROCESSED_DIR
# Import pipeline symbols from the module (not the package) and reload so a long-lived
# kernel picks up new names like prune_all after you git pull or edit code.
import src.preprocessing.pipeline as _pipeline
importlib.reload(_pipeline)
from src.preprocessing.pipeline import (
    clean_all,
    flatten_all,
    load_all_raw,
    preprocess_all,
    prune_all,
    screen_uninformative_all,
    reduce_all,
    transform_all,
    write_processed,
)
from src.preprocessing.column_lineage import build_lineage_all, write_column_lineage
from src.preprocessing.config import default_config
from src.preprocessing.eda import (
    dataset_overview,
    duplicate_report_all,
    null_report,
)
from src.spark.session import create_spark_session

In [3]:
# Optional: set *before* create_spark_session. After changing, run spark.stop() then create_spark_session again
# (kernel restart not required). Then re-run pipeline cells — DataFrames from the old session are invalid.
import os
# os.environ["SPARK_LOCAL_DIRS"] = "/var/tmp/spark-scratch"  # only if project/home disk is also quota-limited (default is artifacts/spark_local_scratch)
# os.environ["SPARK_MAX_CORES"] = "1"  # local[1] — lowest CPU load
# os.environ["SPARK_DRIVER_MEMORY"] = "768m"
# os.environ["SPARK_SQL_SHUFFLE_PARTITIONS"] = "4"
# os.environ["YELP_WRITE_PARTITIONS"] = "32"  # more Parquet shards → smaller per-task sorts

In [4]:
spark = create_spark_session("run_preprocessing")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 20:38:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/29 20:38:41 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


**Pipeline:** Run cells in order: 1 load, 2 clean, 3 flatten, 4 prune, **4b uninformative screen**, 5 transform, 6 reduce, 7 write Parquet + `column_lineage.json`. Long steps log a heartbeat every 60s.

**Resources:** `create_spark_session` defaults to `local[2]`, 1g driver memory, bounded shuffle partitions, and **`artifacts/spark_local_scratch`** for shuffle/spill (not `/tmp`). Override **`SPARK_LOCAL_DIRS`** if that path is quota-limited. For Parquet writes, set `YELP_WRITE_PARTITIONS`. `nice -n 10` for Jupyter can help under load.


In [5]:
# Stage 1: Load raw data (progress logged per dataset; heartbeat every 60s during long steps)
raw = load_all_raw(spark)

[21:38:44] Stage 1: Loading raw data (6 datasets)
[21:38:44]   [1/6] business: loading...
[21:38:46]   [1/6] business: done (150346 rows, 2.3s)
[21:38:46]   [2/6] review: loading...


[21:38:54]   [2/6] review: done (6990280 rows, 8.2s)
[21:38:54]   [3/6] user: loading...


[21:38:58]   [3/6] user: done (1987897 rows, 4.3s)
[21:38:58]   [4/6] checkin: loading...


[21:38:59]   [4/6] checkin: done (131930 rows, 0.3s)
[21:38:59]   [5/6] tip: loading...
[21:38:59]   [5/6] tip: done (908915 rows, 0.5s)
[21:38:59]   [6/6] photo: loading...
[21:39:00]   [6/6] photo: done (200100 rows, 0.2s)
[21:39:00] Stage 1: finished.


---
### After Stage 1 (raw)

`dataset_overview`, duplicate keys, null fractions. For large tables (`review`) set `EDA_SAMPLE_FRACTION = 0.1` below to speed up null counts.


In [6]:
# EDA_SAMPLE_FRACTION=None — full scan; use 0.1 for faster demo on review
EDA_SAMPLE_FRACTION = None

display(dataset_overview(raw))
display(duplicate_report_all(raw))
for name in raw:
    print(f"\n=== null_report: {name} (sample={EDA_SAMPLE_FRACTION}) ===")
    display(null_report(raw[name], name, sample_fraction=EDA_SAMPLE_FRACTION))


,table,rows,columns
0,business,150346,14
1,review,6990280,9
2,user,1987897,22
3,checkin,131930,2
4,tip,908915,5
5,photo,200100,4


,table,total_rows,distinct_key_rows,duplicate_rows,keys
0,business,150346,150346,0,business_id
1,review,6990280,6990280,0,review_id
2,user,1987897,1987897,0,user_id
3,checkin,131930,131930,0,"business_id,date"
4,tip,908915,908836,79,"user_id,business_id,date"
5,photo,200100,200098,2,photo_id



=== null_report: business (sample=None) ===


,column,n_rows,null_count,null_fraction,table
12,categories,150346,150346,1.000000,business
13,hours,150346,23223,0.154464,business
11,attributes,150346,13744,0.091416,business
2,address,150346,0,0.000000,business
0,business_id,150346,0,0.000000,business
1,name,150346,0,0.000000,business
5,postal_code,150346,0,0.000000,business
4,state,150346,0,0.000000,business
3,city,150346,0,0.000000,business
6,latitude,150346,0,0.000000,business



=== null_report: review (sample=None) ===


...

,column,n_rows,null_count,null_fraction,table
0,business_id,131930,0,0.0,checkin
1,date,131930,0,0.0,checkin



=== null_report: tip (sample=None) ===


,column,n_rows,null_count,null_fraction,table
0,text,908915,0,0.0,tip
1,date,908915,0,0.0,tip
2,compliment_count,908915,0,0.0,tip
3,business_id,908915,0,0.0,tip
4,user_id,908915,0,0.0,tip



=== null_report: photo (sample=None) ===


,column,n_rows,null_count,null_fraction,table
0,photo_id,200100,0,0.0,photo
1,business_id,200100,0,0.0,photo
2,caption,200100,0,0.0,photo
3,label,200100,0,0.0,photo


In [7]:
# Stage 2: Clean (nulls, duplicates)
cleaned = clean_all(spark, raw)

[21:40:49] Stage 2: Cleaning (6 datasets)
[21:40:49]   [1/6] business: cleaning...
[21:40:52]   [1/6] business: done (150346 rows, 3.2s)
[21:40:52]   [2/6] review: cleaning...


[21:42:01]   [2/6] review: done (6990280 rows, 68.9s)
[21:42:01]   [3/6] user: cleaning...


[21:43:01]   ... still running: clean user


[21:43:33]   [3/6] user: done (1987897 rows, 91.8s)
[21:43:33]   [4/6] checkin: cleaning...


[21:43:35]   [4/6] checkin: done (131930 rows, 2.0s)
[21:43:35]   [5/6] tip: cleaning...


[21:43:38]   [5/6] tip: done (908836 rows, 3.0s)
[21:43:38]   [6/6] photo: cleaning...


[21:43:38]   [6/6] photo: done (200098 rows, 0.3s)
[21:43:38] Stage 2: finished.


---
### After Stage 2 (cleaned)

Same reports as above on `cleaned` — row counts, duplicate keys, null fractions after imputation/dedup.


In [8]:
display(dataset_overview(cleaned))
display(duplicate_report_all(cleaned))
for name in cleaned:
    print(f"\n=== null_report (cleaned): {name} ===")
    display(null_report(cleaned[name], name, sample_fraction=EDA_SAMPLE_FRACTION))


,table,rows,columns
0,business,150346,14
1,review,6990280,9
2,user,1987897,22
3,checkin,131930,2
4,tip,908836,5
5,photo,200098,4


,table,total_rows,distinct_key_rows,duplicate_rows,keys
0,business,150346,150346,0,business_id
1,review,6990280,6990280,0,review_id
2,user,1987897,1987897,0,user_id
3,checkin,131930,131930,0,"business_id,date"
4,tip,908836,908836,0,"user_id,business_id,date"
5,photo,200098,200098,0,photo_id



=== null_report (cleaned): business ===


,column,n_rows,null_count,null_fraction,table
12,categories,150346,150346,1.000000,business
13,hours,150346,23223,0.154464,business
11,attributes,150346,13744,0.091416,business
2,address,150346,0,0.000000,business
0,business_id,150346,0,0.000000,business
1,name,150346,0,0.000000,business
5,postal_code,150346,0,0.000000,business
4,state,150346,0,0.000000,business
3,city,150346,0,0.000000,business
6,latitude,150346,0,0.000000,business



=== null_report (cleaned): review ===


...

,column,n_rows,null_count,null_fraction,table
0,business_id,131930,0,0.0,checkin
1,date,131930,0,0.0,checkin



=== null_report (cleaned): tip ===


,column,n_rows,null_count,null_fraction,table
0,text,908836,0,0.0,tip
1,date,908836,0,0.0,tip
2,compliment_count,908836,0,0.0,tip
3,business_id,908836,0,0.0,tip
4,user_id,908836,0,0.0,tip



=== null_report (cleaned): photo ===


,column,n_rows,null_count,null_fraction,table
0,photo_id,200098,0,0.0,photo
1,business_id,200098,0,0.0,photo
2,caption,200098,0,0.0,photo
3,label,200098,0,0.0,photo


In [9]:
# Stage 3: Flatten nested columns (attributes, hours, friends, elite, checkin date)
flattened = flatten_all(cleaned)

[21:45:59] Stage 3: Flattening (6 datasets)
[21:45:59]   [1/6] business: flattening...
[21:46:00]   [1/6] business: done (150346 rows, 0.7s)
[21:46:00]   [2/6] review: flattening...


[21:46:12]   [2/6] review: done (6990280 rows, 11.8s)
[21:46:12]   [3/6] user: flattening...


[21:46:18]   [3/6] user: done (1987897 rows, 6.4s)
[21:46:18]   [4/6] checkin: flattening...


[21:46:20]   [4/6] checkin: done (131930 rows, 1.7s)
[21:46:20]   [5/6] tip: flattening...


[21:46:22]   [5/6] tip: done (908836 rows, 1.8s)
[21:46:22]   [6/6] photo: flattening...


[21:46:22]   [6/6] photo: done (200098 rows, 0.3s)
[21:46:22] Stage 3: finished.


In [10]:
# Stage 4: Prune redundant nested/array columns after flatten
pruned = prune_all(spark, flattened)


[21:46:22] Stage 4: Prune after flatten (6 datasets)
[21:46:22]   [1/6] business: pruning...
[21:46:22]   [1/6] business: done (150346 rows, 0.4s)
[21:46:22]   [2/6] review: pruning...


[21:46:36]   [2/6] review: done (6990280 rows, 13.3s)
[21:46:36]   [3/6] user: pruning...


[21:46:41]   [3/6] user: done (1987897 rows, 5.6s)
[21:46:41]   [4/6] checkin: pruning...


[21:46:43]   [4/6] checkin: done (131930 rows, 1.5s)
[21:46:43]   [5/6] tip: pruning...


[21:46:44]   [5/6] tip: done (908836 rows, 1.7s)
[21:46:44]   [6/6] photo: pruning...


[21:46:45]   [6/6] photo: done (200098 rows, 0.3s)
[21:46:45] Stage 4: finished.


In [11]:
# Stage 4b: Drop uninformative columns (all-null or constant); see config drop_uninformative
screened, dropped_uninformative = screen_uninformative_all(spark, pruned)


[21:46:45] Stage 4b: Uninformative screen (6 datasets)
[21:46:45]   [1/6] business: screening...


[21:46:55]   [1/6] business: done (150346 rows, 2 dropped, 10.0s)
[21:46:55]   [2/6] review: screening...


[21:47:55]   ... still running: screen review


[21:48:55]   ... still running: screen review


[21:49:55]   ... still running: screen review


...

[21:55:07]   [2/6] review: done (6990280 rows, 1 dropped, 492.2s)
[21:55:07]   [3/6] user: screening...


[21:55:35]   [3/6] user: done (1987897 rows, 3 dropped, 28.1s)
[21:55:35]   [4/6] checkin: screening...


[21:55:41]   [4/6] checkin: done (131930 rows, 0 dropped, 5.5s)
[21:55:41]   [5/6] tip: screening...


[21:55:48]   [5/6] tip: done (908836 rows, 0 dropped, 7.2s)
[21:55:48]   [6/6] photo: screening...


[21:55:49]   [6/6] photo: done (200098 rows, 0 dropped, 1.0s)
[21:55:49] Stage 4b: finished.


In [12]:
# Stage 5: Transform (scale, parse dates, drop raw date strings after parsing)
transformed = transform_all(spark, screened)


[21:55:49] Stage 5: Transforming (6 datasets)
[21:55:49]   [1/6] business: transforming...
[21:55:52]   [1/6] business: done (150346 rows, 3.1s)
[21:55:52]   [2/6] review: transforming...


[21:57:03]   [2/6] review: done (6990280 rows, 70.8s)
[21:57:03]   [3/6] user: transforming...


[21:58:03]   ... still running: transform user


[21:58:56]   [3/6] user: done (1987897 rows, 113.6s)
[21:58:56]   [4/6] checkin: transforming...


[21:59:01]   [4/6] checkin: done (131930 rows, 4.6s)
[21:59:01]   [5/6] tip: transforming...


[21:59:05]   [5/6] tip: done (908836 rows, 3.7s)
[21:59:05]   [6/6] photo: transforming...


[21:59:05]   [6/6] photo: done (200098 rows, 0.3s)
[21:59:05] Stage 5: finished.


In [13]:
# Stage 6: Reduce (optional sampling)
processed = reduce_all(spark, transformed)

[21:59:05] Stage 6: Reduce (6 datasets)
[21:59:05]   [1/6] business: reduce...
[21:59:05]   [1/6] business: done (150346 rows, 0.4s)
[21:59:05]   [2/6] review: reduce...


[21:59:17]   [2/6] review: done (6990280 rows, 12.0s)
[21:59:17]   [3/6] user: reduce...


[21:59:23]   [3/6] user: done (1987897 rows, 5.9s)
[21:59:23]   [4/6] checkin: reduce...


[21:59:25]   [4/6] checkin: done (131930 rows, 1.7s)
[21:59:25]   [5/6] tip: reduce...


[21:59:27]   [5/6] tip: done (908836 rows, 1.7s)
[21:59:27]   [6/6] photo: reduce...


[21:59:27]   [6/6] photo: done (200098 rows, 0.3s)
[21:59:27] Stage 6: finished.


In [14]:
# Stage 7: Write Parquet and column lineage JSON (artifacts/processed/column_lineage.json)
write_processed(processed)
cols_cleaned = {n: list(df.columns) for n, df in cleaned.items()}
cols_flat = {n: list(df.columns) for n, df in flattened.items()}
cols_pruned = {n: list(df.columns) for n, df in pruned.items()}
cols_final = {n: list(df.columns) for n, df in processed.items()}
lineage = build_lineage_all(
    cols_cleaned,
    cols_flat,
    cols_pruned,
    cols_final,
    default_config(),
    dropped_uninformative=dropped_uninformative,
)
write_column_lineage(lineage)


[21:59:27] Writing Parquet (6 datasets) to /home/dash/University/term-6/big-data/yelp-spark-project/artifacts/processed (16 partitions each)
[21:59:27]   [1/6] business: writing...


[21:59:35]   [1/6] business: done (8.1s)
[21:59:35]   [2/6] review: writing...


[22:09:20]   [2/6] review: done (584.8s)
[22:09:20]   [3/6] user: writing...


[22:09:53]   [3/6] user: done (33.3s)
[22:09:53]   [4/6] checkin: writing...


[22:10:06]   [4/6] checkin: done (12.5s)
[22:10:06]   [5/6] tip: writing...


[22:10:10]   [5/6] tip: done (4.6s)
[22:10:10]   [6/6] photo: writing...


[22:10:11]   [6/6] photo: done (1.0s)
[22:10:11] Writing: finished.


PosixPath('/home/dash/University/term-6/big-data/yelp-spark-project/artifacts/processed/column_lineage.json')

---
### Outputs and next steps

- Processed data: `artifacts/processed/<table>/` (Parquet).
- Column lineage (including `dropped_uninformative_columns`): `artifacts/processed/column_lineage.json`.
- Detailed comparison and numeric summaries: `preprocessing_comparison.ipynb`.
